# 🤖 WorldQuant BRAIN Alpha Mining Bot - AutoBrain Sim

LangGraph notebook for alpha discovery using `autobrain-sim` for WorldQuant submission.

In [8]:
from typing import TypedDict
import os
import json
import time
from pathlib import Path

import httpx
from langgraph.graph import END, START, StateGraph
from IPython.display import Image, display
from brain_client import BrainClient

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

if load_dotenv is not None:
    load_dotenv('.env')

In [ ]:
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY', '')
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1/chat/completions'

WQ_USERNAME = os.getenv('WQ_USERNAME', '')
WQ_PASSWORD = os.getenv('WQ_PASSWORD', '')

SHARPE_TARGET = float(os.getenv('SHARPE_TARGET', '1.25'))
MAX_ALPHA_ITERATIONS = int(os.getenv('MAX_ALPHA_ITERATIONS', '20'))
SUCCESSFUL_ALPHAS_FILE = Path('successful_alphas.json')

OPENROUTER_MAX_RETRIES = int(os.getenv('OPENROUTER_MAX_RETRIES', '3'))
OPENROUTER_BASE_BACKOFF_SECONDS = float(os.getenv('OPENROUTER_BASE_BACKOFF_SECONDS', '1'))
OPENROUTER_REQUEST_PACING_SECONDS = float(os.getenv('OPENROUTER_REQUEST_PACING_SECONDS', '0.5'))

MODEL_MAP = {
    'hypothesis': 'openai/gpt-oss-120b:free',
    'coder': 'openai/gpt-oss-120b:free',
    'validator': 'openai/gpt-oss-120b:free',
    'analyzer': 'openai/gpt-oss-120b:free',
    'embedder': 'openai/gpt-oss-120b:free',
}

MODEL_FALLBACKS = {
    'hypothesis': ['openai/gpt-oss-120b:free'],
    'coder': ['openai/gpt-oss-120b:free'],
    'validator': ['openai/gpt-oss-120b:free'],
    'analyzer': ['openai/gpt-oss-120b:free'],
    'embedder': ['openai/gpt-oss-120b:free'],
}

In [10]:
class AlphaMiningState(TypedDict):
    hypothesis: str
    alpha_expression: str
    validation_status: bool
    backtest_results: dict
    feedback: str
    iteration_count: int

In [11]:
def load_worldquant_meta_database(file_path='worldquant_meta_database.json'):
    path = Path(file_path)
    if not path.exists():
        return []

    with path.open('r', encoding='utf-8') as handle:
        data = json.load(handle)

    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        return data.get('fields', []) or data.get('data', []) or []
    return []


def search_relevant_fields(keyword, limit=5, file_path='worldquant_meta_database.json'):
    metadata = load_worldquant_meta_database(file_path)
    keyword = keyword.lower().strip()
    matches = []

    if not keyword:
        return ['close', 'open', 'high', 'low', 'volume'][:limit]

    for item in metadata:
        if not isinstance(item, dict):
            continue
        searchable_text = ' '.join(str(item.get(key, '')).lower() for key in ('name', 'definition', 'description', 'category', 'documentation'))
        if keyword in searchable_text:
            field_name = str(item.get('name', '')).strip()
            if field_name and field_name not in matches:
                matches.append(field_name)
        if len(matches) >= limit:
            break

    return matches[:limit] if matches else ['close', 'open', 'high', 'low', 'volume'][:limit]


def call_openrouter(agent_key, user_prompt, system_prompt='You are a helpful quant assistant.'):
    primary_model = MODEL_MAP[agent_key]
    fallback_models = MODEL_FALLBACKS.get(agent_key, [primary_model])
    models_to_try = []
    for model in [primary_model] + fallback_models:
        if model not in models_to_try:
            models_to_try.append(model)

    if not OPENROUTER_API_KEY:
        return f'mock_response_from_{agent_key}'

    headers = {
        'Authorization': f'Bearer {OPENROUTER_API_KEY}',
        'Content-Type': 'application/json',
    }

    for model_name in models_to_try:
        for attempt in range(OPENROUTER_MAX_RETRIES):
            payload = {
                'model': model_name,
                'messages': [
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': user_prompt},
                ],
                'temperature': 0.2,
            }

            try:
                response = httpx.post(OPENROUTER_BASE_URL, headers=headers, json=payload, timeout=45.0)
                if response.status_code == 429:
                    wait_time = OPENROUTER_BASE_BACKOFF_SECONDS * (2 ** attempt)
                    print(f'⏳ Rate limit ({agent_key}, {model_name}) - retry in {wait_time:.1f}s')
                    time.sleep(wait_time)
                    continue
                response.raise_for_status()
                content = response.json().get('choices', [{}])[0].get('message', {}).get('content', '').strip()
                time.sleep(OPENROUTER_REQUEST_PACING_SECONDS)
                return content or f'mock_response_from_{agent_key}'
            except Exception as exc:
                print(f'⚠️ OpenRouter Error ({agent_key}): {exc}')
                if attempt < OPENROUTER_MAX_RETRIES - 1:
                    time.sleep(OPENROUTER_BASE_BACKOFF_SECONDS * (2 ** attempt))
                else:
                    print(f'↪️ Switching model for {agent_key} after retries: {model_name}')

    return f'mock_response_from_{agent_key}'

In [12]:
def get_brain_client():
    if WQ_USERNAME and WQ_PASSWORD:
        return BrainClient.login(WQ_USERNAME, WQ_PASSWORD)
    return BrainClient()


def generate_hypothesis(state):
    print(f"Running generate_hypothesis with model: {MODEL_MAP['hypothesis']}")
    iteration_count = state.get('iteration_count', 0) + 1
    feedback = state.get('feedback', '').strip()
    prompt = f"Generate one concise alpha hypothesis. Feedback: {feedback or 'none'}"
    hypothesis_text = call_openrouter('hypothesis', prompt, 'You generate alpha research hypotheses.')
    print(f"💡 New Hypothesis: {hypothesis_text}")
    return {'hypothesis': hypothesis_text, 'iteration_count': iteration_count}


def code_alpha(state):
    print(f"Running code_alpha with model: {MODEL_MAP['coder']}")
    hypothesis = state.get('hypothesis', '')
    fields = search_relevant_fields(hypothesis)
    field_hint = ', '.join(fields)
    prompt = (
        f"Hypothesis: {hypothesis}\n"
        f"Available fields: {field_hint}\n"
        'CRITICAL RULE: You must write ONLY A SINGLE, continuous inline mathematical expression.\n'
        'NO variable assignments. NO newlines. NO semicolons. NO markdown tags.\n'
        'EXAMPLE GOOD 1: rank(ts_sum(close, 30) / ts_mean(volume, 20))\n'
        'EXAMPLE GOOD 2: (close - ts_delay(close, 1)) / close\n'
        'OUTPUT THE EXPRESSION ONLY:'
    )
    alpha_expression = call_openrouter('coder', prompt, 'You are an expert WorldQuant FASTEXPR coder.')
    alpha_expression = alpha_expression.replace('```python', '').replace('```alpha', '').replace('```', '').strip()
    alpha_expression = ' '.join(alpha_expression.splitlines())
    print(f"💻 Generated Alpha: {alpha_expression}")
    return {'alpha_expression': alpha_expression, 'validation_status': False}


def validate_alpha(state):
    print(f"Running validate_alpha with model: {MODEL_MAP['validator']}")
    alpha_expression = state.get('alpha_expression', '').strip()
    is_valid = True
    feedback_msg = ''
    if not alpha_expression:
        is_valid = False
        feedback_msg = 'Error: Expression is empty.'
    elif '\n' in alpha_expression or '=' in alpha_expression or ';' in alpha_expression:
        is_valid = False
        feedback_msg = 'Error: Variable assignments (=), semicolons (;), and newlines are STRICTLY FORBIDDEN. Write a single nested expression.'
    else:
        _ = call_openrouter('validator', f'Check this alpha expression syntax: {alpha_expression}', 'You validate alpha syntax quickly.')
        feedback_msg = ''
    print(f"🔎 Validation Passed: {is_valid} | Feedback: {feedback_msg}")
    return {'validation_status': is_valid, 'feedback': feedback_msg if not is_valid else ''}


def run_backtest(state):
    print('Running run_backtest')
    expression = state.get('alpha_expression', '').strip()

    fail_result = {'backtest_results': {'sharpe': 0.0, 'turnover': 0.0, 'fitness': 0.0}}

    if not expression:
        print('📊 Backtest Results -> Sharpe: 0.0, Fitness: 0.0, Turnover: 0.0')
        return fail_result

    if not WQ_USERNAME or not WQ_PASSWORD:
        print('⚠️ WQ Credentials missing!')
        print('📊 Backtest Results -> Sharpe: 0.0, Fitness: 0.0, Turnover: 0.0')
        return fail_result

    settings = {
        'instrumentType': 'EQUITY',
        'region': 'USA',
        'universe': 'TOP3000',
        'delay': 1,
        'decay': 2,
        'neutralization': 'SUBINDUSTRY',
        'truncation': 0.08,
        'pasteurization': 'ON',
        'unitHandling': 'VERIFY',
        'nanHandling': 'ON',
        'language': 'FASTEXPR',
        'visualization': False,
    }

    try:
        client = get_brain_client()
        sim = client.simulate(expression=expression, settings=settings)
        wait_result = sim.wait(verbose=True)
        alpha_id = sim.alpha_id or (wait_result.get('alpha') if isinstance(wait_result, dict) else None)

        if not alpha_id:
            raise RuntimeError('AutoBrain did not return an alpha id after wait().')

        alpha = client.get_alpha(alpha_id)
        is_data = alpha.get('is', {}) if isinstance(alpha, dict) else {}
        metrics = {
            'sharpe': float(is_data.get('sharpe', 0) or 0),
            'turnover': float(is_data.get('turnover', 0) or 0),
            'fitness': float(is_data.get('fitness', 0) or 0),
        }
        print(f"📊 Backtest Results -> Sharpe: {metrics.get('sharpe')}, Fitness: {metrics.get('fitness')}, Turnover: {metrics.get('turnover')}")
        return {'backtest_results': metrics}
    except Exception as exc:
        print(f'⚠️ Autobrain Error: {exc}')
        print('📊 Backtest Results -> Sharpe: 0.0, Fitness: 0.0, Turnover: 0.0')
        return fail_result


def analyze_results(state):
    print(f"Running analyze_results with model: {MODEL_MAP['analyzer']}")
    sharpe = float(state.get('backtest_results', {}).get('sharpe', 0))
    if sharpe > SHARPE_TARGET:
        success_entry = {
            'alpha_expression': state.get('alpha_expression', ''),
            'hypothesis': state.get('hypothesis', ''),
            'backtest_results': state.get('backtest_results', {}),
        }
        try:
            if SUCCESSFUL_ALPHAS_FILE.exists():
                with SUCCESSFUL_ALPHAS_FILE.open('r', encoding='utf-8') as infile:
                    existing = json.load(infile)
                if not isinstance(existing, list):
                    existing = []
            else:
                existing = []
        except Exception:
            existing = []
        existing.append(success_entry)
        with SUCCESSFUL_ALPHAS_FILE.open('w', encoding='utf-8') as outfile:
            json.dump(existing, outfile, ensure_ascii=False, indent=2)
        print(f"✅ Saved successful alpha to {SUCCESSFUL_ALPHAS_FILE}")
    _ = call_openrouter('analyzer', f'Analyze result with sharpe={sharpe} and suggest next action.', 'You analyze alpha backtest metrics.')
    feedback = '' if sharpe > SHARPE_TARGET else 'Sharpe below target. Try a new hypothesis.'
    print(f"🧠 Analyzer Feedback: {feedback}")
    return {'feedback': feedback}

In [13]:
def route_after_validation(state):
    if not state.get('validation_status', False):
        return 'code_alpha'
    return 'run_backtest'


def route_after_analysis(state):
    if state.get('iteration_count', 0) >= MAX_ALPHA_ITERATIONS:
        return END
    return 'generate_hypothesis'


workflow = StateGraph(AlphaMiningState)
workflow.add_node('generate_hypothesis', generate_hypothesis)
workflow.add_node('code_alpha', code_alpha)
workflow.add_node('validate_alpha', validate_alpha)
workflow.add_node('run_backtest', run_backtest)
workflow.add_node('analyze_results', analyze_results)
workflow.add_edge(START, 'generate_hypothesis')
workflow.add_edge('generate_hypothesis', 'code_alpha')
workflow.add_edge('code_alpha', 'validate_alpha')
workflow.add_conditional_edges('validate_alpha', route_after_validation, {'code_alpha': 'code_alpha', 'run_backtest': 'run_backtest'})
workflow.add_edge('run_backtest', 'analyze_results')
workflow.add_conditional_edges('analyze_results', route_after_analysis, {'generate_hypothesis': 'generate_hypothesis', END: END})
app = workflow.compile()

In [14]:
initial_state = {
    'hypothesis': '',
    'alpha_expression': '',
    'validation_status': False,
    'backtest_results': {},
    'feedback': '',
    'iteration_count': 0,
}

config = {'recursion_limit': 1000}

final_state = app.invoke(initial_state, config=config)
print(json.dumps(final_state, indent=2, ensure_ascii=False))

Running generate_hypothesis with model: openai/gpt-oss-120b:free
⏳ Rate limit (hypothesis, openai/gpt-oss-120b:free) - retry in 1.0s
⏳ Rate limit (hypothesis, openai/gpt-oss-120b:free) - retry in 2.0s
⏳ Rate limit (hypothesis, openai/gpt-oss-120b:free) - retry in 4.0s
💡 New Hypothesis: mock_response_from_hypothesis
Running code_alpha with model: openai/gpt-oss-120b:free
⏳ Rate limit (coder, openai/gpt-oss-120b:free) - retry in 1.0s
⏳ Rate limit (coder, openai/gpt-oss-120b:free) - retry in 2.0s
⏳ Rate limit (coder, openai/gpt-oss-120b:free) - retry in 4.0s
💻 Generated Alpha: mock_response_from_coder
Running validate_alpha with model: openai/gpt-oss-120b:free
⏳ Rate limit (validator, openai/gpt-oss-120b:free) - retry in 1.0s
⏳ Rate limit (validator, openai/gpt-oss-120b:free) - retry in 2.0s
⏳ Rate limit (validator, openai/gpt-oss-120b:free) - retry in 4.0s
🔎 Validation Passed: True | Feedback: 
Running run_backtest
Simulating... sleeping for 5.0 seconds
Alpha simulation complete. Alpha I

KeyboardInterrupt: 

In [ ]:
display(Image(app.get_graph().draw_mermaid_png()))